# ST-OMR Meter V5-3K — Background Feature / Domain Shift Forensics

TRAIN-only, read-only representation/domain-shift analysis. **LAUNCH only once.** If the browser disconnects, reconnect and use STATUS only. No training, threshold tuning, Historical Validation, First-30, V5 VAL, or FINAL_HOLDOUT access.

In [ ]:
from pathlib import Path
from datetime import datetime, timezone
import hashlib, json, os, shutil, subprocess, sys, time

RUNNER_HEAD = "5c132192dd0949377b4b7291a0e12c970b2cbec1"
RUNNER_BLOB = "4ff07ec4238642d6140c80dc9b544692dfb415b7"
FORENSICS_IMPLEMENTATION_HEAD = "0efa6b3ba315d671e5a449789ff6de103c735439"
V53G_HEAD = "b36a9d2f5daade2c3568cac8cbc736ca75ca435f"
EXPECTED_V53G_REPORT_SHA256 = "682c2d405287051fef18b803e2597777cb7fc55c6ba0814ea3b2d4df0fa35b9d"
EXPECTED_V53H_ENVELOPE_SHA256 = "f41b0fddb9d139018e0ddd16c9765d9415031e6308efd67e16aef3a05d205bf7"
EXPECTED_V53J_REPORT_SHA256 = "7a49d29e0d7257be7c59d499ab3d9ab575d369a7473b0b5298ea62aa80c7d37f"
REPOSITORY = "khfy7wpr5p-maker/st-omr-training"
RUNNER_REL = "tools/meter_v5_3k_background_runner_v1.py"
MYDRIVE = Path("/content/drive/MyDrive")

if not MYDRIVE.is_dir():
    from google.colab import drive
    drive.mount("/content/drive")

DATA_ROOT = MYDRIVE / "TEST" / "METER_V2_1500_PACKAGE_AB_CLEAN"
CHECKPOINT_ROOT = MYDRIVE / "ST-OMR-METER-SPECIALISTS"
M4A_ROOT = CHECKPOINT_ROOT / "m4a-234-digit-specialist-dataset-freeze-v2"
D10_ROOT = MYDRIVE / "ST-OMR-D10" / "stage7d10-authoritative-562c8fcfabf1b41573f1ef591d88ae65335ce16a"
for name, path in {"DATA_ROOT": DATA_ROOT, "CHECKPOINT_ROOT": CHECKPOINT_ROOT, "M4A_ROOT": M4A_ROOT, "D10_ROOT": D10_ROOT}.items():
    if not path.is_dir():
        raise RuntimeError(f"{name} not found: {path}")
print("DRIVE/PATH CHECK = PASS")

ANN = DATA_ROOT / "annotations"
SOURCE_REPORT = ANN / "v5_3g_authoritative_rescue_training_report.json"
SOURCE_ENVELOPE = ANN / f"v5_3g_execution_envelope_{V53G_HEAD}.json"
V53J_REPORT = ANN / "v5_3j_rescue_failure_forensics_v1.json"
RESCUE_DIR = ANN / "v5_3g_authoritative_rescue_artifacts"
for name, path in {"V5-3G REPORT": SOURCE_REPORT, "V5-3H ENVELOPE": SOURCE_ENVELOPE, "V5-3J REPORT": V53J_REPORT}.items():
    if not path.is_file() or path.is_symlink():
        raise RuntimeError(f"{name} missing/non-regular: {path}")
if not RESCUE_DIR.is_dir() or RESCUE_DIR.is_symlink():
    raise RuntimeError(f"RESCUE DIR missing/non-regular: {RESCUE_DIR}")

def sha_file(path):
    h = hashlib.sha256()
    with Path(path).open("rb") as handle:
        for block in iter(lambda: handle.read(1024 * 1024), b""):
            h.update(block)
    return h.hexdigest()

if sha_file(SOURCE_REPORT) != EXPECTED_V53G_REPORT_SHA256:
    raise RuntimeError("V5-3G report SHA mismatch")
if sha_file(SOURCE_ENVELOPE) != EXPECTED_V53H_ENVELOPE_SHA256:
    raise RuntimeError("V5-3H envelope SHA mismatch")
if sha_file(V53J_REPORT) != EXPECTED_V53J_REPORT_SHA256:
    raise RuntimeError("V5-3J report SHA mismatch")
print("SOURCE EVIDENCE CHECK = PASS")
print("V5-3J REPORT SHA =", EXPECTED_V53J_REPORT_SHA256)

CONTROL_DIR = ANN / "v5_3k_background_control"
CONTROL_DIR.mkdir(parents=True, exist_ok=True)
LOCK = CONTROL_DIR / f"launch_{FORENSICS_IMPLEMENTATION_HEAD}.json"
HEARTBEAT = CONTROL_DIR / f"heartbeat_{FORENSICS_IMPLEMENTATION_HEAD}.json"
PROGRESS = CONTROL_DIR / f"progress_{FORENSICS_IMPLEMENTATION_HEAD}.json"
LOG = CONTROL_DIR / f"background_{FORENSICS_IMPLEMENTATION_HEAD}.log"
RESULT = ANN / "v5_3k_feature_domain_shift_forensics_v1.json"

if RESULT.exists():
    raise RuntimeError(f"Existing V5-3K report blocks launch: {RESULT}")
if LOCK.exists():
    state = json.loads(LOCK.read_text(encoding="utf-8"))
    raise RuntimeError("V5-3K launch lock already exists; second process is forbidden. " f"status={state.get('status')} pid={state.get('pid')}")
print("OUTPUT/LOCK GUARD = PASS")

SOURCE_REPO = Path("/content/st-omr-v5-3k-runner-source")
repo_url = f"https://github.com/{REPOSITORY}.git"
if SOURCE_REPO.exists():
    shutil.rmtree(SOURCE_REPO)
subprocess.check_call(["git", "clone", "--no-checkout", repo_url, str(SOURCE_REPO)])
subprocess.check_call(["git", "-C", str(SOURCE_REPO), "fetch", "origin", RUNNER_HEAD, "--depth", "1"])
fetched = subprocess.check_output(["git", "-C", str(SOURCE_REPO), "rev-parse", "FETCH_HEAD"], text=True).strip()
if fetched != RUNNER_HEAD:
    raise RuntimeError(f"runner FETCH_HEAD mismatch: {fetched}")
subprocess.check_call(["git", "-C", str(SOURCE_REPO), "checkout", "--detach", RUNNER_HEAD])
actual_head = subprocess.check_output(["git", "-C", str(SOURCE_REPO), "rev-parse", "HEAD"], text=True).strip()
if actual_head != RUNNER_HEAD:
    raise RuntimeError(f"runner HEAD mismatch: {actual_head}")
if subprocess.check_output(["git", "-C", str(SOURCE_REPO), "status", "--porcelain"], text=True).strip():
    raise RuntimeError("runner source worktree dirty")
runner_path = SOURCE_REPO / RUNNER_REL
if not runner_path.is_file():
    raise RuntimeError(f"runner missing: {runner_path}")
actual_blob = subprocess.check_output(["git", "-C", str(SOURCE_REPO), "hash-object", RUNNER_REL], text=True).strip()
if actual_blob != RUNNER_BLOB:
    raise RuntimeError(f"runner blob mismatch: {actual_blob}")
runner_source = runner_path.read_text(encoding="utf-8")
compile(runner_source, str(runner_path), "exec")
if runner_source.count("forensics.run_feature_domain_shift_forensics_v1(") != 1:
    raise RuntimeError("runner forensics-call count changed")
for forbidden in ("run_authoritative_rescue_training_v1(", "execute_rescue_tensor_harness_v1(", "run_historical_retention_gate(", "torch.optim.", ".backward(", "optimizer.step("):
    if forbidden in runner_source:
        raise RuntimeError(f"runner contains forbidden training/validation token: {forbidden}")
print("EXACT BACKGROUND RUNNER = PASS")
print("RUNNER HEAD =", RUNNER_HEAD)
print("RUNNER BLOB =", RUNNER_BLOB)

initial = {
    "schema": "st-omr-meter-v5-3k-background-launch-v1",
    "forensics_implementation_head": FORENSICS_IMPLEMENTATION_HEAD,
    "runner_head": RUNNER_HEAD,
    "runner_blob": RUNNER_BLOB,
    "v5_3j_report_sha256": EXPECTED_V53J_REPORT_SHA256,
    "status": "ALLOCATED",
    "allocated_at_utc": datetime.now(timezone.utc).isoformat(),
    "log_path": str(LOG),
    "heartbeat_path": str(HEARTBEAT),
    "progress_path": str(PROGRESS),
    "result_path": str(RESULT),
}
payload = (json.dumps(initial, indent=2, sort_keys=True) + "\n").encode("utf-8")
fd = os.open(str(LOCK), os.O_WRONLY | os.O_CREAT | os.O_EXCL, 0o600)
try:
    os.write(fd, payload)
finally:
    os.close(fd)

log_handle = LOG.open("ab", buffering=0)
env = os.environ.copy()
env["PYTHONUNBUFFERED"] = "1"
try:
    proc = subprocess.Popen([sys.executable, "-u", str(runner_path)], stdin=subprocess.DEVNULL, stdout=log_handle, stderr=subprocess.STDOUT, start_new_session=True, close_fds=True, env=env)
finally:
    log_handle.close()

time.sleep(2)
if proc.poll() is not None:
    tail = LOG.read_text(encoding="utf-8", errors="replace").splitlines()[-120:]
    raise RuntimeError("V5-3K background runner exited immediately:\n" + "\n".join(tail))
state = json.loads(LOCK.read_text(encoding="utf-8"))
print("V5-3K BACKGROUND LAUNCH = PASS")
print("PID =", state.get("pid", proc.pid))
print("STATUS =", state.get("status"))
print("LOG =", LOG)
print("HEARTBEAT =", HEARTBEAT)
print("PROGRESS =", PROGRESS)
print("RESULT =", RESULT)
print("If the browser disconnects, do not run LAUNCH again; use STATUS only.")

In [ ]:
from pathlib import Path
from datetime import datetime, timezone
import json

FORENSICS_IMPLEMENTATION_HEAD = "0efa6b3ba315d671e5a449789ff6de103c735439"
ANN = Path("/content/drive/MyDrive/TEST/METER_V2_1500_PACKAGE_AB_CLEAN/annotations")
CONTROL_DIR = ANN / "v5_3k_background_control"
LOCK = CONTROL_DIR / f"launch_{FORENSICS_IMPLEMENTATION_HEAD}.json"
HEARTBEAT = CONTROL_DIR / f"heartbeat_{FORENSICS_IMPLEMENTATION_HEAD}.json"
PROGRESS = CONTROL_DIR / f"progress_{FORENSICS_IMPLEMENTATION_HEAD}.json"
LOG = CONTROL_DIR / f"background_{FORENSICS_IMPLEMENTATION_HEAD}.log"
RESULT = ANN / "v5_3k_feature_domain_shift_forensics_v1.json"

if not LOCK.is_file():
    raise RuntimeError("V5-3K launch receipt not found.")
state = json.loads(LOCK.read_text(encoding="utf-8"))
print("STATUS =", state.get("status"))
print("PID =", state.get("pid"))
print("ALLOCATED =", state.get("allocated_at_utc"))
print("STARTED =", state.get("started_at_utc"))
print("COMPLETED =", state.get("completed_at_utc"))
print("ERROR =", state.get("error_type"), state.get("error_message"))
print("RESULT EXISTS =", RESULT.is_file())
if HEARTBEAT.is_file():
    hb = json.loads(HEARTBEAT.read_text(encoding="utf-8"))
    print("HEARTBEAT =", hb)
    if hb.get("utc"):
        age = (datetime.now(timezone.utc) - datetime.fromisoformat(hb["utc"])).total_seconds()
        print("HEARTBEAT AGE SEC =", round(age, 1))
if PROGRESS.is_file():
    p = json.loads(PROGRESS.read_text(encoding="utf-8"))
    print("PROGRESS =", p)
    total = int(p.get("total") or 0)
    done = int(p.get("done") or 0)
    if total > 0:
        print("PROGRESS PCT =", round(100.0 * done / total, 2))
if LOG.is_file():
    lines = LOG.read_text(encoding="utf-8", errors="replace").splitlines()
    print("\n--- LOG TAIL (last 140 lines) ---")
    print("\n".join(lines[-140:]))

In [ ]:
from pathlib import Path
import hashlib, json

FORENSICS_IMPLEMENTATION_HEAD = "0efa6b3ba315d671e5a449789ff6de103c735439"
ANN = Path("/content/drive/MyDrive/TEST/METER_V2_1500_PACKAGE_AB_CLEAN/annotations")
RESULT = ANN / "v5_3k_feature_domain_shift_forensics_v1.json"
LOCK = ANN / "v5_3k_background_control" / f"launch_{FORENSICS_IMPLEMENTATION_HEAD}.json"

if not RESULT.is_file():
    state = json.loads(LOCK.read_text(encoding="utf-8")) if LOCK.is_file() else {}
    print("FINAL FORENSICS RECEIPT = NOT READY")
    print("BACKGROUND STATUS =", state.get("status"))
else:
    raw = RESULT.read_bytes()
    report = json.loads(raw.decode("utf-8"))
    digest = hashlib.sha256(raw).hexdigest()
    state = json.loads(LOCK.read_text(encoding="utf-8")) if LOCK.is_file() else {}
    print("FINAL FORENSICS RECEIPT = READY")
    print("BACKGROUND STATUS =", state.get("status"))
    print("DIAGNOSIS SCOPE =", report.get("diagnosis_scope"))
    for digit in ("2", "3"):
        item = report["per_specialist"][digit]
        witness = item["fixed_threshold_witness"]
        same_neg = item["same_label_negative_domain_shift"]
        critical = item["critical_v5_positive_vs_historical_tn"]
        hard = item["historical_hard_tn_subpopulation"]
        print(f"\n=== {digit}-AI ===")
        print("V5-3J FAILURE SIGNATURE =", item["v5_3j_failure_signature"])
        print("V5 POSITIVE RECOVERY FRACTION =", witness["v5_positive_recovery_fraction"])
        print("HISTORICAL HARD TN COUNT =", witness["historical_hard_tn_count"])
        print("SAME-LABEL NEG 64D CENTROID/RMS =", same_neg["feature_64d"]["geometry"]["centroid_distance_over_sum_within_rms"])
        print("SAME-LABEL NEG 8D LOGIT GAP =", same_neg["output_logit_gap_decomposition"]["mean_logit_gap_a_minus_b"])
        print("CRITICAL V5POS-vs-HISTTN 64D CENTROID/RMS =", critical["feature_64d"]["geometry"]["centroid_distance_over_sum_within_rms"])
        print("CRITICAL 64D MAX STD SHIFT =", critical["feature_64d"]["standardized_mean_shift"]["max_absolute_standardized_mean_shift"])
        print("CRITICAL 8D MAX STD SHIFT =", critical["hidden_8d"]["standardized_mean_shift"]["max_absolute_standardized_mean_shift"])
        print("CRITICAL 8D MEAN LOGIT GAP =", critical["output_logit_gap_decomposition"]["mean_logit_gap_a_minus_b"])
        print("CRITICAL TOP 64D DIMS =", critical["feature_64d"]["standardized_mean_shift"]["top_dimensions"][:5])
        print("CRITICAL TOP HIDDEN CONTRIBUTIONS =", critical["output_logit_gap_decomposition"]["hidden_dimension_contributions"][:4])
        print("HARD-vs-PRESERVED HIST TN 64D CENTROID/RMS =", hard["feature_64d"]["geometry"]["centroid_distance_over_sum_within_rms"])
        print("HARD-vs-PRESERVED HIST TN TOP 64D DIMS =", hard["feature_64d"]["standardized_mean_shift"]["top_dimensions"][:5])
        print("GROUP IDENTITY REVERIFIED =", item["group_identity_reverified"])
    print("\nFROZEN STATE BIT IDENTICAL =", report.get("frozen_state_bit_identical"))
    print("RESCUE STATE BIT IDENTICAL DURING FORENSICS =", report.get("rescue_state_bit_identical_during_forensics"))
    print("REPAIR RECIPE SELECTED =", report.get("repair_recipe_selected"))
    print("RETRAINING AUTHORIZED =", report.get("retraining_authorized"))
    print("HISTORICAL VALIDATION OPENED =", report.get("historical_validation_opened"))
    print("FIRST-30 OPENED =", report.get("first30_opened"))
    print("V5 RESERVE OPENED =", report.get("v5_reserve_opened"))
    print("V5 VALIDATION OPENED =", report.get("v5_validation_opened"))
    print("FINAL_HOLDOUT LOCKED =", report.get("final_holdout_locked"))
    print("V5-3K REPORT SHA256 =", digest)